In [10]:
# ===== CONFIGURATION =====
import pandas as pd
import re
import numpy as np
import logging
import os
from pathlib import Path
from typing import Optional, Union

# Load configuration from .env file (path is hidden)
from config_loader import DATA_RAW_DIR, CLEANED_DATA_DIR

In [ ]:
# LOAD DATA
res_2023 = pd.read_excel(DATA_RAW_DIR / "res_2023.xlsx").ffill(axis=0)
res_2024 = pd.read_excel(DATA_RAW_DIR / "res_2024.xlsx").ffill(axis=0)
res_2025 = pd.read_excel(DATA_RAW_DIR / "res_2025.xlsx").ffill(axis=0)


# Set the years
res_2023['year'] = 2023
res_2024['year'] = 2024
res_2025['year'] = 2025

# Concatenate all years
residence = pd.concat([res_2023, res_2024, res_2025], ignore_index=True)
print(f"✓ Loaded {len(residence):,} property records")

✓ Loaded 68,680 property records


In [ ]:
# Concatenate the DataFrames row-wise
master_df = residence.drop(columns=['Unit', 'Unit        '])

master_df.columns = master_df.columns.str.strip().str.lower().str.replace(' ', '_').reset_index(drop=True)

# Replace commas and convert to float
master_df['main_floor_area'] = master_df['main_floor_area'].replace({',': '', '-': 0}, regex=True).astype(float)
master_df['land/parcel_area'] = master_df['land/parcel_area'].replace({',': '', '-': 0}, regex=True).astype(float)
master_df['transaction_price'] = master_df['transaction_price'].replace({'RM': '', ',': ''}, regex=True).astype(float)


# Identify rows where 'Main Floor Area' is greater than 'Land/Parcel Area'
condition = master_df['main_floor_area'] > master_df['land/parcel_area']

# For these identified rows, swap the values between the two columns
master_df.loc[master_df['main_floor_area'] == 0, 'main_floor_area'] = master_df.loc[master_df['main_floor_area'] == 0, 'land/parcel_area']
master_df.loc[condition, ['main_floor_area', 'land/parcel_area']] = master_df.loc[condition, ['land/parcel_area', 'main_floor_area']].values

<bound method IndexOpsMixin.tolist of Index(['property_type', 'district', 'mukim', 'scheme_name/area', 'road_name',
       'month,_year_of_transaction_date', 'tenure', 'land/parcel_area',
       'main_floor_area', 'unit_level', 'transaction_price', 'year'],
      dtype='object')>


In [3]:
master_df.describe()

,land/parcel_area,main_floor_area,transaction_price,year
count,68680.000000,68680.000000,6.868000e+04,68680.000000
mean,175.494538,121.831289,6.900488e+05,2023.461080
std,2942.023377,74.883830,8.109327e+05,0.555696
min,12.000000,12.000000,2.000000e+04,2023.000000
25%,85.000000,77.000000,3.100000e+05,2023.000000
50%,128.000000,105.000000,4.800000e+05,2023.000000
75%,176.000000,142.000000,7.820000e+05,2024.000000
max,728424.000000,4570.810000,3.800000e+07,2025.000000


In [4]:
def clean_and_process_addresses(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans, standardizes, and enriches address data in a DataFrame.

    This function performs the following steps:
    1.  Converts specified address columns to uppercase and strips whitespace.
    2.  Adds a boolean flag 'unidentified_road_name' to identify rows where the road name was originally missing.
    3.  Derives the state ('WP Kuala Lumpur', 'WP Putrajaya', 'Selangor') from the 'district' column.
    4.  Extracts road names from 'scheme_name/area' to fill in the missing 'road_name' values.
    5.  If 'road_name' is still missing, it uses the 'scheme_name/area' as a fallback.
    6.  Standardizes common abbreviations (e.g., 'JLN' to 'JALAN') in the final 'road_name' column.

    Args:
        df: A pandas DataFrame with address columns like 'district', 'road_name', and 'scheme_name/area'.

    Returns:
        A pandas DataFrame with cleaned and processed address information.
    """
    # Create a copy to avoid modifying the original DataFrame
    df = df.copy()

    # --- 1. Initial Cleaning ---
    # Convert specified columns to uppercase and strip leading/trailing whitespace.
    address_cols = ['district', 'mukim', 'scheme_name/area', 'road_name']
    for col in address_cols:
        if col in df.columns:
            df[col] = df[col].str.strip().str.upper()

    # --- 2. Flag Unidentified Road Names (Based on Original Data) ---
    # Replace empty strings with NaN for consistent handling of missing values.
    df['road_name'].replace('', np.nan, inplace=True)
    # Flag rows where road_name was originally missing.
    df['unidentified_road_name'] = df['road_name'].isna()

    # --- 3. Derive State from District ---
    # Use np.select for efficient conditional assignment based on district name.
    conditions = [
        df['district'].str.contains('KUALA LUMPUR', na=False),
        df['district'].str.contains('PUTRAJAYA', na=False)
    ]
    choices = ['WP KUALA LUMPUR', 'WP PUTRAJAYA']
    df['state'] = np.select(conditions, choices, default='Selangor')

    # --- 4. Extract and Fill Missing Road Names ---
    # Define a regex pattern to find common road prefixes and the text that follows.
    road_prefixes = [
        'TAMAN', 'JALAN', 'LEBUH', 'LRT', 'LORONG',
        'LINTANG', 'SOLOK', 'BANDAR', 'KAMPUNG'
    ]
    extraction_pattern = re.compile(
        r'\b(' + '|'.join(road_prefixes) + r')\b[^\(\)\n]*',
        flags=re.IGNORECASE
    )

    def extract_road_from_scheme(scheme_name: str) -> str:
        """Helper function to apply the regex pattern to a single string."""
        if pd.isna(scheme_name):
            return np.nan
        match = extraction_pattern.search(scheme_name)
        return match.group(0).strip() if match else np.nan

    # Apply the extraction function only on rows that were flagged as unidentified.
    missing_road_mask = df['unidentified_road_name']
    df.loc[missing_road_mask, 'road_name'] = df.loc[missing_road_mask, 'scheme_name/area'].apply(extract_road_from_scheme)

    # If road_name is still NaN after extraction, use the scheme_name/area as a fallback.
    df['road_name'].fillna(df['scheme_name/area'], inplace=True)

    # --- 5. Standardize All Address Abbreviations Last ---
    # Consolidate all replacements into a single dictionary for efficiency.
    road_name_replacements = {
        r'\b-?(?:JLN|JALN|JALANG|JLAN|OFF\s+JALAN|JALANLAN|OFJLN|JA;LAN|JALAN\.|J\.?)\b': 'JALAN',
        r'\bTMN\b': 'TAMAN',
        r'\b(OFF PERSIARAN|PERSIRN|PRSRN\.?)\b': 'PERSIARAN',
        r'\bLTG\b': 'LINTANG',
        r'\bBKT\b': 'BUKIT',
        r'\bSG\b': 'SUNGAI',
        r'\bKG\b': 'KAMPUNG',
        r'\bSLK\b': 'SOLOK',
        r'\bKLN\b': 'KILANG',
        r'\bLEBOH\b': 'LEBUH',
        # WARNING: This regex is broad and might incorrectly capture acronyms like 'P.J.'
        # Consider making it more specific if it causes issues.
        r'\b(?:[A-Z]\.\s*)+': 'LORONG',
    }

    # Apply replacements to the now-filled road_name column.
    df['road_name'] = df['road_name'].replace(road_name_replacements, regex=True)

    return df



# Process the DataFrame using the refactored function
master_df = clean_and_process_addresses(master_df)

In [5]:
import math

def clean_unit_level(unit_level_series: pd.Series) -> pd.Series:
    """
    Cleans and converts a pandas Series of unit/level strings to integers.

    This function handles various formats including:
    - Standard numbers ('5', '12')
    - Ground floors ('G', 'UG', 'LG', 'P')
    - Ranges ('1-4', '2&3'), taking the average
    - Excel date errors ('1-Mar' -> treated as '1-3')
    - Levels with letters ('3A', '13A'), extracting the number
    - Large invalid numbers (like Excel serial dates), converting them to 0
    - Invalid or empty values, converting them to 0

    Args:
        unit_level_series: A pandas Series containing the messy unit level strings.

    Returns:
        A pandas Series of the same length with cleaned integer values.
    """

    def convert_level(level):
        # 1. Handle non-string or empty/whitespace values first
        if not isinstance(level, str) or not level.strip():
            return 0

        level = level.strip().upper()

        # 2. Handle special non-numeric levels
        level_mapping = {
            'G': 0,   # Ground Floor
            'P': 0,   # Podium/Parking, often synonymous with Ground
            'LG': 0, # Lower Ground
            'UG': 0,  # Upper Ground, often same as Ground
            'MZ': 0,  # Mezzanine, often between G and 1
            'T': 0    # Terrace/Top, treated as 0
        }
        if level in level_mapping:
            return level_mapping[level]

        # 3. Handle Excel date format error (e.g., '1-MAR')
        month_map = {
            'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6,
            'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12
        }
        date_error_match = re.match(r'^(\d+)-([A-Z]{3})$', level)
        if date_error_match:
            day, month_str = date_error_match.groups()
            if month_str in month_map:
                start, end = int(day), month_map[month_str]
                return math.ceil((start + end) / 2)

        # 4. Handle ranges (e.g., '1-4', '2&3', '2-3')
        range_match = re.match(r'^(\d+)\s*[-&]\s*(\d+)$', level)
        if range_match:
            start, end = map(int, range_match.groups())
            return math.ceil((start + end) / 2)

        # 5. Handle decimals (e.g., '1.5', '2.5')
        if '.' in level:
            try:
                return math.ceil(float(level))
            except (ValueError, TypeError):
                pass # Continue to next check

        # 6. Handle numbers, potentially with trailing letters (e.g., '3A', '45718')
        numeric_part_match = re.match(r'^(\d+)', level)
        if numeric_part_match:
            try:
                num = int(numeric_part_match.group(1))
                # Treat huge numbers (likely Excel errors) as invalid
                if num > 1000:
                    return 0
                return num
            except (ValueError, TypeError):
                return 0

        # 7. If none of the above, it's an invalid format
        return 0

    # Apply the conversion function to each element in the series
    cleaned_series = unit_level_series.apply(convert_level).astype(int)
    
    return cleaned_series

In [6]:
# Apply the cleaning function
master_df['unit_level_cleaned'] = clean_unit_level(master_df['unit_level'])

master_df['unit_level_cleaned'].value_counts()

unit_level_cleaned
0     40434
1      3573
2      3461
3      3192
4      2467
5      1770
6      1204
8      1175
7      1119
9      1041
10     1017
11      869
12      830
13      783
14      627
15      621
16      524
17      480
18      416
19      340
20      298
21      274
22      255
23      225
25      163
24      155
26      153
27      149
28      109
29      106
31      100
32       92
33       91
30       87
35       80
34       78
36       75
37       52
38       46
39       45
40       25
41       19
43       15
44       10
42       10
45        9
46        5
51        3
53        2
47        2
55        1
48        1
57        1
50        1
Name: count, dtype: int64

In [7]:
master_df

,property_type,district,mukim,scheme_name/area,road_name,"month,_year_of_transaction_date",tenure,land/parcel_area,main_floor_area,unit_level,transaction_price,year,unidentified_road_name,state,unit_level_cleaned
0,1 - 1 1/2 Storey Semi-Detached,GOMBAK,BANDAR SELAYANG,TAMAN SELAYANG MUTIARA,JALAN MUTIARA 3/8A,October 2023,Leasehold,201.0,74.0,,315000.0,2023,False,Selangor,0
1,1 - 1 1/2 Storey Semi-Detached,GOMBAK,BANDAR SELAYANG,TAMAN SELAYANG MUTIARA,JALAN MUTIARA 4/4A,December 2023,Leasehold,205.0,74.0,,440000.0,2023,False,Selangor,0
2,1 - 1 1/2 Storey Semi-Detached,GOMBAK,BANDAR SELAYANG,TAMAN SELAYANG MUTIARA,JALAN MUTIARA 5/2,December 2023,Leasehold,205.0,74.0,,570000.0,2023,False,Selangor,0
3,1 - 1 1/2 Storey Semi-Detached,GOMBAK,BANDAR SELAYANG,TAMAN SELAYANG MUTIARA,JALAN 43,September 2023,Leasehold,224.0,70.0,,465000.0,2023,False,Selangor,0
4,1 - 1 1/2 Storey Semi-Detached,GOMBAK,BANDAR SELAYANG,TAMAN SELAYANG MUTIARA,JALAN MUTIARA 3/3,September 2023,Leasehold,303.0,74.0,,590000.0,2023,False,Selangor,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68675,Town House,PETALING,PEKAN KAYU ARA,KG SUNGAI KAYU ARA,KAMPUNG SUNGAI KAYU ARA,February 2025,Leasehold,125.0,125.0,G,900000.0,2025,True,Selangor,0
68676,Town House,PETALING,PEKAN KAYU ARA,KG SUNGAI KAYU ARA,KAMPUNG SUNGAI KAYU ARA,February 2025,Leasehold,127.0,127.0,2,500000.0,2025,True,Selangor,2
68677,Town House,SEPANG,DENGKIL,BANDAR SIERRA PUCHONG (ODORA PARKHOMES),BANDAR SIERRA PUCHONG,January 2025,Leasehold,181.0,181.0,1&2,710000.0,2025,True,Selangor,2
68678,Town House,SEPANG,DENGKIL,KOTA WARISAN,KOTA WARISAN,January 2025,Freehold,164.0,164.0,2-3,480000.0,2025,True,Selangor,3


In [ ]:
master_df.to_csv(CLEANED_DATA_DIR / "df_v1.csv", index=False)